# 🎨 SD Image Generator
Stable Diffusion XL · Balanced profile · SDXL-native sizes · Long prompt support · Face mode

In [ ]:
import subprocess, sys, os

os.environ["HF_HOME"] = "/kaggle/working/hf_cache"

def install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

PACKAGES = [
    "diffusers",
    "accelerate",
    "transformers",
    "huggingface_hub",
    "ipywidgets",
    "insightface",
    "onnxruntime-gpu",
    "opencv-python-headless",
]
for pkg in PACKAGES:
    install(pkg)

import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
import random, io, base64
from PIL import Image, ImageOps

print(f"✅ Ready | PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
STYLE_MODELS = {
    "Realism":       "SG161222/RealVisXL_V4.0",
    "Anime":         "cagliostrolab/animagine-xl-3.1",
    "Comics":        "Lykon/dreamshaper-xl-1-0",
    "Illustration":  "playgroundai/playground-v2.5-1024px-aesthetic",
    "Lineart":       "stabilityai/stable-diffusion-xl-base-1.0",
}

# SDXL-native sizes that match YouTube aspect ratios exactly
# Generated at these sizes, displayed as-is (no upscale)
SIZES = {
    "YouTube Video (16:9)":  (1152, 640),
    "YouTube Shorts (9:16)": (640, 1152),
}

# Balanced profile constants
BASE_STEPS     = 40
DEFAULT_GUIDANCE = 6.5

DEFAULT_NEGATIVE = (
    "deformed, ugly, blurry, low quality, worst quality, duplicate, multiple people, "
    "extra limbs, extra arms, extra hands, extra fingers, fused fingers, malformed hands, "
    "bad anatomy, bad hands, mutated hands, poorly drawn hands, wrong hands, missing fingers"
)

import gc
from diffusers import DiffusionPipeline
from huggingface_hub import file_exists

def release_resources(*pipelines):
    for p in pipelines:
        if p is None:
            continue
        try:
            p.unload_ip_adapter()
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_pipeline(model_id: str):
    try:
        has_diffusers = file_exists(model_id, "model_index.json")
    except Exception:
        has_diffusers = False

    if not has_diffusers:
        raise ValueError(f"{model_id} has no model_index.json — only diffusers-format models supported")

    try:
        pipe = DiffusionPipeline.from_pretrained(
            model_id, torch_dtype=torch.float16,
            use_safetensors=True, variant="fp16",
        )
    except Exception:
        pipe = DiffusionPipeline.from_pretrained(
            model_id, torch_dtype=torch.float16, use_safetensors=True,
        )

    pipe.to("cuda")

    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

    return pipe

pipe = None

style_radio = widgets.RadioButtons(
    options=list(STYLE_MODELS.keys()),
    description="Style:",
    layout=widgets.Layout(width="400px"),
)
load_btn = widgets.Button(
    description="⚙️ Load model",
    button_style="primary",
    layout=widgets.Layout(width="200px"),
)
model_status = widgets.Label(value="No model loaded")

def on_load(btn):
    global pipe
    model_status.value = f"⏳ Loading {style_radio.value}..."
    load_btn.disabled = True
    old_pipe = pipe
    pipe = None
    release_resources(old_pipe)
    try:
        pipe = load_pipeline(STYLE_MODELS[style_radio.value])
        model_status.value = f"✅ {style_radio.value} ready"
    except Exception as e:
        model_status.value = f"❌ {e}"
        pipe = None
    finally:
        load_btn.disabled = False

load_btn.on_click(on_load)
display(widgets.VBox([
    widgets.Label("🎨 Select style:"),
    style_radio,
    widgets.HBox([load_btn, model_status]),
]))

In [ ]:
from PIL import ImageOps

# ── Gallery ──────────────────────────────────────────────────────────────────

def image_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

def show_gallery(images, seeds_used):
    cards = []
    for i, (img, seed) in enumerate(zip(images, seeds_used)):
        b64 = image_to_b64(img)
        dl = widgets.HTML(
            f'<a download="img_{i+1}_seed{seed}.png" href="data:image/png;base64,{b64}">' +
            f'<button>⬇ Download</button></a>'
        )
        w, h = img.size
        thumb_w, thumb_h = (320, 180) if w > h else (180, 320)
        thumb = widgets.Image(
            value=base64.b64decode(b64), format="png",
            width=thumb_w, height=thumb_h,
        )
        cards.append(widgets.VBox([thumb, dl]))
    cols = 4 if len(cards) > 2 else len(cards)
    rows = [widgets.HBox(cards[i:i+cols]) for i in range(0, len(cards), cols)]
    display(widgets.VBox(rows))

# ── Tokenizer / Compel ───────────────────────────────────────────────────────

def count_tokens(pipe, text: str) -> int:
    """Count CLIP tokens for the given text using pipe.tokenizer."""
    try:
        tokens = pipe.tokenizer(
            text,
            truncation=False,
            return_tensors="pt",
        ).input_ids
        return tokens.shape[-1] - 2  # subtract BOS and EOS
    except Exception:
        return 0

def encode_prompt(pipe, prompt: str, negative_prompt: str, use_compel: bool = True) -> dict:
    """Always returns plain strings — compel removed due to IP-Adapter incompatibility."""
    return {"prompt": prompt, "negative_prompt": negative_prompt}

# ── Generation ───────────────────────────────────────────────────────────────

def make_generator(seed: int):
    return torch.Generator(device="cuda").manual_seed(int(seed))

def run_pipeline(pipe, prompt_kwargs: dict, width: int, height: int,
                  guidance: float, seed: int, face_img=None) -> Image.Image:
    generator = make_generator(seed)
    common = {
        **prompt_kwargs,
        "width": width,
        "height": height,
        "generator": generator,
        "guidance_scale": guidance,
        "num_images_per_prompt": 1,
    }
    if face_img is not None:
        common["ip_adapter_image"] = face_img

    return pipe(
        num_inference_steps=BASE_STEPS,
        **common,
    ).images[0]

def load_face_adapter(pipe, face_bytes: bytes, scale: float):
    face_img = ImageOps.exif_transpose(
        Image.open(io.BytesIO(face_bytes))
    ).convert("RGB")
    face_img.thumbnail((1024, 1024))
    if not getattr(pipe, "_ip_adapter_loaded", False):
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        pipe.load_ip_adapter(
            "h94/IP-Adapter",
            subfolder="sdxl_models",
            weight_name="ip-adapter-plus-face_sdxl_vit-h.safetensors",
            image_encoder_folder="models/image_encoder",
        )
        pipe._ip_adapter_loaded = True
    pipe.set_ip_adapter_scale(scale)
    return face_img

def get_face_bytes(upload_value):
    if not upload_value:
        return None
    if isinstance(upload_value, dict):
        return next(iter(upload_value.values()))["content"]
    first = upload_value[0]
    return first["content"] if isinstance(first, dict) else first.content

# ── Widgets ──────────────────────────────────────────────────────────────────

prompt_ta = widgets.Textarea(
    placeholder="Describe what you want to generate...",
    description="Prompt:",
    layout=widgets.Layout(width="600px", height="80px"),
)
token_label = widgets.Label(value="Tokens: 0 / 77")

neg_prompt_ta = widgets.Textarea(
    value=DEFAULT_NEGATIVE,
    description="Negative:",
    layout=widgets.Layout(width="600px", height="60px"),
)
count_slider = widgets.IntSlider(
    value=2, min=1, max=8, step=1,
    description="Count:",
    layout=widgets.Layout(width="400px"),
)
guidance_slider = widgets.FloatSlider(
    value=DEFAULT_GUIDANCE, min=1.0, max=12.0, step=0.5,
    description="Guidance:",
    layout=widgets.Layout(width="400px"),
)
seed_input = widgets.IntText(
    value=42, description="Seed:",
    layout=widgets.Layout(width="200px"),
)
random_seed_btn = widgets.Button(
    description="🎲 Random",
    layout=widgets.Layout(width="120px"),
)
seed_mode_radio = widgets.RadioButtons(
    options=["Fixed", "Random each time"],
    value="Fixed",
    description="Seed mode:",
)
size_radio = widgets.RadioButtons(
    options=list(SIZES.keys()),
    description="Size:",
)
face_upload = widgets.FileUpload(
    accept="image/*",
    description="Face (opt.):",
    layout=widgets.Layout(width="300px"),
)
face_scale_slider = widgets.FloatSlider(
    value=0.7, min=0.0, max=1.0, step=0.05,
    description="Face scale:",
    layout=widgets.Layout(width="400px"),
)
generate_btn = widgets.Button(
    description="🚀 Generate",
    button_style="success",
    layout=widgets.Layout(width="200px", height="40px"),
)
progress_bar = widgets.IntProgress(
    value=0, min=0, max=100,
    description="Progress:",
    layout=widgets.Layout(width="400px"),
)
status_label = widgets.Label(value="Ready")
gallery_output = widgets.Output()

# ── Token counter (live update) ──────────────────────────────────────────────

def update_token_count(change):
    if pipe is None:
        token_label.value = "Tokens: — (load model first)"
        return
    try:
        n = count_tokens(pipe, prompt_ta.value)
        over = n > 77
        token_label.value = f"Tokens: {n} / 77{'  ⚠️ truncated at 77 (compel extends this)' if over else ''}"
    except Exception:
        token_label.value = "Tokens: —"

prompt_ta.observe(update_token_count, names="value")

# ── Handlers ─────────────────────────────────────────────────────────────────

def on_random_seed(btn):
    seed_input.value = random.randint(0, 2**32 - 1)
random_seed_btn.on_click(on_random_seed)

def on_generate(btn):
    if pipe is None:
        status_label.value = "❌ Load a model first"
        return

    prompt = prompt_ta.value.strip()
    if not prompt:
        status_label.value = "❌ Prompt is empty"
        return

    generate_btn.disabled = True
    progress_bar.value = 0
    status_label.value = "⏳ Preparing..."

    try:
        negative_prompt = neg_prompt_ta.value.strip() or DEFAULT_NEGATIVE
        width, height = SIZES[size_radio.value]
        guidance = guidance_slider.value
        count = count_slider.value

        # Build seed list
        if seed_mode_radio.value == "Fixed":
            base_seed = int(seed_input.value)
            seeds = [base_seed] * count
        else:
            seeds = [random.randint(0, 2**32 - 1) for _ in range(count)]
            seed_input.value = seeds[0]

        # Encode prompt — compel for text-only, plain strings for face mode
        # (compel and IP-Adapter are incompatible)
        status_label.value = "⏳ Encoding prompt..."
        face_bytes = get_face_bytes(face_upload.value)
        use_compel = face_bytes is None
        prompt_kwargs = encode_prompt(pipe, prompt, negative_prompt, use_compel=use_compel)

        # Face
        face_img = None
        if face_bytes:
            status_label.value = "⏳ Loading face adapter..."
            face_img = load_face_adapter(pipe, face_bytes, face_scale_slider.value)
        elif getattr(pipe, "_ip_adapter_loaded", False):
            pipe.unload_ip_adapter()
            pipe._ip_adapter_loaded = False

        # Generate loop
        images = []
        for i, seed in enumerate(seeds):
            status_label.value = f"⏳ Generating {i+1}/{count} (seed {seed})..."
            progress_bar.value = int(10 + 80 * i / count)
            img = run_pipeline(
                pipe, prompt_kwargs, width, height, guidance, seed, face_img,
            )
            images.append(img)

        progress_bar.value = 95
        with gallery_output:
            clear_output(wait=True)
            show_gallery(images, seeds)

        progress_bar.value = 100
        seed_str = str(seeds[0]) if len(set(seeds)) == 1 else f"{seeds[0]}..{seeds[-1]}"
        status_label.value = f"✅ Done — {len(images)} image(s) | seed: {seed_str}"

    except Exception as e:
        status_label.value = f"❌ Error: {e}"
    finally:
        generate_btn.disabled = False

generate_btn.on_click(on_generate)

# ── Layout ───────────────────────────────────────────────────────────────────

display(widgets.VBox([
    prompt_ta,
    token_label,
    neg_prompt_ta,
    widgets.HBox([count_slider, guidance_slider]),
    widgets.HBox([seed_input, random_seed_btn]),
    seed_mode_radio,
    size_radio,
    face_upload,
    face_scale_slider,
    generate_btn,
    widgets.HBox([progress_bar, status_label]),
]))
display(gallery_output)